# Parte 3 · Fine-tuning A e B, comparação, Grad-CAM e teste

**Projeto de CNN · Machine Learning e Deep Learning (CC6NA 2026.2) · Profa. Polyana Nascimento**
Equipe Inception (vaga 1): Alberto Acosta, Davi Corrêa, Gabriel Albuquerque

Este notebook continua o `02_dados_e_base_congelada.ipynb` e cobre:

| Seção | Conteúdo |
|---|---|
| 2 | **Planejamento** das duas estratégias de fine-tuning (o que muda, o que fica fixo e por quê) |
| 3–4 | Execução da **estratégia A** e da **estratégia B** |
| 5 | **Comparação** das três condições na validação (métricas, parâmetros treináveis, épocas, tempo, curvas, matrizes de confusão) |
| 6 | Análise visual nas **mesmas imagens** (erros corrigidos, persistentes e novos) |
| 7 | **Grad-CAM** nas mesmas imagens nos três modelos |
| 8 | Escolha do modelo **pela validação** e, só então, avaliação dos três no **teste** (etapa 7 da lauda) |
| 9 | Modelo salvo, carregado sem retreinar e usado numa **imagem da internet** |

**Entradas** (geradas pelo notebook 02, na pasta `artifacts/`): `split.csv`, `classes.json`, `modelo_base_congelada.pt`, `history_base_congelada.csv`, `val_preds_base_congelada.csv`, `resumo_base_congelada.json`.

**Ambiente:** Google Colab com GPU (T4). Em uma sessão nova do Colab os artefatos da parte 2 precisam estar no Drive (`USE_DRIVE`).

## 0. Ambiente e configuração

In [ ]:
# No Colab, torch/torchvision/sklearn já vêm instalados; timm e kagglehub às vezes não.
%pip install -q timm kagglehub

In [ ]:
import os, sys, json, time, copy, random, shutil, platform, io, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms as T
import timm

from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix)

ImageFile.LOAD_TRUNCATED_IMAGES = False
sns.set_theme(style="whitegrid")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python", platform.python_version(), "| torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| timm", timm.__version__)
print("Dispositivo:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

In [ ]:
# ---------------- Configuração ----------------
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"   # só para testar o código rapidamente em CPU

# Fixo nas três condições (igual ao notebook 02)
SEED = 42
BATCH_SIZE = 8 if SMOKE_TEST else 32
NUM_WORKERS = 0 if SMOKE_TEST else 2
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 2 if SMOKE_TEST else 20   # orçamento máximo de épocas por condição
PATIENCE = 4
MONITOR = "val_f1_macro"

# Estratégia A: topo descongelado, etapa única
LR_A = 1e-4

# Estratégia B: descongelamento progressivo + learning rate menor nos blocos
B_ETAPA1_EPOCAS = 1 if SMOKE_TEST else 5   # etapa 1 (topo); a etapa 2 usa o restante do orçamento
LR_B_CLASSIFICADOR = 1e-4
LR_B_BLOCOS = 1e-5

# Blocos de model.features no timm: 0–2 stem, 3–5 Mixed, 6–9 Inception-A, 10 Reduction-A,
# 11–17 Inception-B, 18 Reduction-B, 19–21 Inception-C
TOPO = slice(18, 22)    # Reduction-B + 3× Inception-C
MEIO = slice(11, 18)    # 7× Inception-B

IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = IN_COLAB and not SMOKE_TEST
ARTIFACTS = Path(os.environ.get("ARTIFACTS_DIR", "artifacts"))
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACTS = Path("/content/drive/MyDrive/cnn-inceptionv4-sea-animals/artifacts")
ARTIFACTS.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything()

NECESSARIOS = ["split.csv", "classes.json", "modelo_base_congelada.pt", "history_base_congelada.csv",
               "val_preds_base_congelada.csv", "resumo_base_congelada.json"]
faltando = [f for f in NECESSARIOS if not (ARTIFACTS / f).exists()]
assert not faltando, f"Faltam artefatos da parte 2 em {ARTIFACTS.resolve()}: {faltando}. Rode o notebook 02 antes."
print("Artefatos em:", ARTIFACTS.resolve())

## 1. Dados, pré-processamento e funções reaproveitadas

Para que as três condições sejam comparáveis, tudo aqui vem da parte 2 **sem alteração**: a mesma divisão (`split.csv`), o mesmo pré-processamento (lido do próprio checkpoint), as mesmas métricas e as mesmas funções de treino (`set_train_mode`, `run_epoch`, `fit`) e de carga (`load_checkpoint`).

In [ ]:
DATA_DIR = os.environ.get("SEA_DATA_DIR")
if DATA_DIR is None:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download("vencerlanz09/sea-animals-image-dataste")
DATA_DIR = Path(DATA_DIR)

split = pd.read_csv(ARTIFACTS / "split.csv")
meta = json.load(open(ARTIFACTS / "classes.json", encoding="utf-8"))
classes = meta["classes"]; NUM_CLASSES = len(classes)
BASE_CKPT = ARTIFACTS / "modelo_base_congelada.pt"

_ck = torch.load(BASE_CKPT, map_location="cpu", weights_only=False)
assert _ck["classes"] == classes, "classes.json e o checkpoint não batem"
DATA_CFG = _ck["data_config"]; MODEL_NAME = _ck["model_name"]; DROP_RATE = _ck["drop_rate"]
del _ck

IMG_SIZE, RESIZE, MEAN, STD = DATA_CFG["img_size"], DATA_CFG["resize"], DATA_CFG["mean"], DATA_CFG["std"]
INTERP = T.InterpolationMode.BICUBIC

faltam_imgs = [p for p in split.path if not (DATA_DIR / p).exists()]
assert not faltam_imgs, f"{len(faltam_imgs)} imagens do split não existem em {DATA_DIR}"
print(f"{NUM_CLASSES} classes | " + " | ".join(f"{s} {n}" for s, n in split.split.value_counts().items()))
print("Pré-processamento:", DATA_CFG)

In [ ]:
def load_rgb(path):
    with Image.open(path) as im:
        if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
            im = im.convert("RGBA")
            bg = Image.new("RGB", im.size, (255, 255, 255))
            bg.paste(im, mask=im.getchannel("A"))
            return bg
        return im.convert("RGB")

def show_grid(paths, titles, ncols=6, size=2.3, suptitle=None):
    n = len(paths)
    nrows = max(1, int(np.ceil(n / ncols)))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * size, nrows * size * 1.25))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    for ax, p, t in zip(np.atleast_1d(axes).ravel(), paths, titles):
        ax.imshow(load_rgb(DATA_DIR / p)); ax.set_title(t, fontsize=7.5)
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout(); plt.show()

train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0), interpolation=INTERP),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([
    T.Resize(RESIZE, interpolation=INTERP),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

def denorm(x):
    return (x * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]).clamp(0, 1).permute(1, 2, 0)

class SeaAnimals(Dataset):
    def __init__(self, frame, root, transform):
        self.paths = frame.path.tolist(); self.ys = frame.y.tolist()
        self.root, self.transform = Path(root), transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.transform(load_rgb(self.root / self.paths[i])), self.ys[i]

def make_loaders(split_df, batch_size=BATCH_SIZE):
    g = torch.Generator().manual_seed(SEED)
    mk = lambda name, tf, shuffle: DataLoader(
        SeaAnimals(split_df[split_df.split == name], DATA_DIR, tf), batch_size=batch_size, shuffle=shuffle,
        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda", generator=g if shuffle else None)
    return mk("train", train_tf, True), mk("val", eval_tf, False), mk("test", eval_tf, False)

def compute_metrics(y_true, y_pred):
    return {
        "acc": accuracy_score(y_true, y_pred),
        "bal_acc": balanced_accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

train_loader, val_loader, test_loader = make_loaders(split)
criterion = nn.CrossEntropyLoss()

In [ ]:
def load_checkpoint(path, device=DEVICE):
    ck = torch.load(path, map_location=device, weights_only=False)
    m = timm.create_model(ck["model_name"], pretrained=False, num_classes=ck["num_classes"], drop_rate=ck["drop_rate"])
    m.load_state_dict(ck["state_dict"])
    return m.to(device), ck

def freeze_base(model):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.get_classifier().parameters():
        p.requires_grad = True

def unfreeze(*modules):
    for mod in modules:
        for p in mod.parameters():
            p.requires_grad = True

def set_train_mode(model):
    # Treino normal, mas BatchNorm sem parâmetros treináveis fica em eval (estatísticas congeladas).
    # Nos blocos descongelados, o BatchNorm volta a treinar.
    model.train()
    for m in model.modules():
        if isinstance(m, nn.modules.batchnorm._BatchNorm) and not any(p.requires_grad for p in m.parameters()):
            m.eval()

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    training = optimizer is not None
    set_train_mode(model) if training else model.eval()
    losses, ys, preds, probs = [], [], [], []
    use_amp = DEVICE.type == "cuda"
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                logits = model(x)
                loss = criterion(logits, y)
            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
            losses.append(loss.item() * len(y))
            p = logits.float().softmax(1)
            ys.append(y.cpu()); preds.append(p.argmax(1).cpu()); probs.append(p.detach().cpu())
    y, yhat = torch.cat(ys).numpy(), torch.cat(preds).numpy()
    return sum(losses) / len(y), y, yhat, torch.cat(probs).numpy()

def fit(model, train_loader, val_loader, optimizer, scheduler=None, max_epochs=MAX_EPOCHS,
        patience=PATIENCE, label="modelo"):
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    best, best_state, best_epoch, wait, hist = -1.0, None, 0, 0, []
    t0 = time.time()
    for epoch in range(1, max_epochs + 1):
        te = time.time()
        tr_loss, ytr, ptr, _ = run_epoch(model, train_loader, criterion, optimizer, scaler)
        va_loss, yva, pva, _ = run_epoch(model, val_loader, criterion)
        mtr, mva = compute_metrics(ytr, ptr), compute_metrics(yva, pva)
        row = {"epoch": epoch, "train_loss": tr_loss, "val_loss": va_loss,
               **{f"train_{k}": v for k, v in mtr.items()}, **{f"val_{k}": v for k, v in mva.items()},
               "lr": optimizer.param_groups[0]["lr"], "seg": time.time() - te}
        hist.append(row)
        if scheduler is not None:
            scheduler.step(row[MONITOR])
        melhorou = row[MONITOR] > best + 1e-4
        if melhorou:
            best, best_epoch, wait = row[MONITOR], epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
        print(f"[{label}] ép {epoch:02d} | loss tr {tr_loss:.4f} va {va_loss:.4f} | F1 tr {mtr['f1_macro']:.4f} "
              f"va {mva['f1_macro']:.4f} | acc va {mva['acc']:.4f} | lr {row['lr']:.1e} | {row['seg']:.0f}s"
              + ("  *" if melhorou else ""))
        if wait >= patience:
            print(f"Early stopping: {patience} épocas sem melhora.")
            break
    model.load_state_dict(best_state)
    hist = pd.DataFrame(hist)
    info = {"best_epoch": best_epoch, "epochs_run": len(hist), "best_" + MONITOR: best,
            "train_time_s": time.time() - t0}
    print(f"Melhor época: {best_epoch} ({MONITOR} = {best:.4f}) | tempo total {info['train_time_s'] / 60:.1f} min")
    return hist, info

def plot_history(hist, title, best_epoch=None, marcos=()):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(hist.epoch, hist.train_loss, "o-", label="treino")
    axes[0].plot(hist.epoch, hist.val_loss, "o-", label="validação")
    axes[0].set_title("Loss (entropia cruzada)"); axes[0].set_xlabel("época")
    axes[1].plot(hist.epoch, hist.train_f1_macro, "o-", label="treino")
    axes[1].plot(hist.epoch, hist.val_f1_macro, "o-", label="validação")
    axes[1].set_title("F1-macro"); axes[1].set_xlabel("época")
    for ax in axes:
        for x, txt in marcos:
            ax.axvline(x, c="darkorange", ls="--", label=txt)
        if best_epoch:
            ax.axvline(best_epoch, c="gray", ls=":", label=f"melhor época ({best_epoch})")
        ax.legend()
    fig.suptitle(title); plt.tight_layout(); plt.show()

In [ ]:
# Conferência: o modelo de referência carregado reproduz as previsões de validação salvas na parte 2
base_model, ck_base = load_checkpoint(BASE_CKPT)
_, yv, pv_base, probv_base = run_epoch(base_model, val_loader, criterion)
val_ref = pd.read_csv(ARTIFACTS / "val_preds_base_congelada.csv")
assert (val_ref.path.values == split[split.split == "val"].path.values).all()
assert (val_ref.pred.values == np.array(classes)[pv_base]).all(), "O checkpoint não reproduz as previsões salvas!"

nomes_blocos = [type(m).__name__ for m in base_model.features]
print("Blocos de features:", nomes_blocos)
assert nomes_blocos[18] == "ReductionB" and all(n == "InceptionC" for n in nomes_blocos[19:22])
assert all(n == "InceptionB" for n in nomes_blocos[11:18])
print("Base congelada OK | F1-macro val:", round(compute_metrics(yv, pv_base)["f1_macro"], 4))

## 2. Planejamento das estratégias de fine-tuning

**Ponto de partida comum.** As duas estratégias começam de **cópias** de `modelo_base_congelada.pt`, isto é, com o classificador novo já treinado. Se descongelássemos a base com um classificador aleatório, os gradientes grandes do início do treino chegariam às camadas pré-treinadas e destruiriam parte do que elas aprenderam no ImageNet.

**Por que descongelar a partir do topo.** Numa CNN, os primeiros blocos aprendem padrões genéricos (bordas, cores, texturas simples), úteis para quase qualquer imagem. Os últimos aprendem combinações específicas do ImageNet (partes de objetos). As fotos subaquáticas têm iluminação azulada, baixo contraste e animais que o ImageNet cobre pouco; é nas camadas finais que a adaptação mais deve ajudar.

### O que fica fixo nas três condições

| Item | Valor |
|---|---|
| Dados | Mesmo `split.csv` (70/15/15), mesmo pré-processamento e aumento de dados |
| Otimização | AdamW (weight decay 1e-4), entropia cruzada, batch 32, precisão mista |
| Scheduler | `ReduceLROnPlateau` (fator 0,5, paciência 2) no F1-macro da validação |
| Parada | Early stopping com paciência 4 no **F1-macro da validação**; orçamento máximo de **20 épocas** por condição |
| Seleção | Pesos da época de maior F1-macro na validação |
| BatchNorm | Congelado (modo avaliação) nos blocos congelados; treina nos blocos descongelados |
| Semente | 42 |

### O que muda

| | **Estratégia A** — topo, etapa única | **Estratégia B** — progressiva, lr diferenciada |
|---|---|---|
| Blocos treinados | `features[18:]` (Reduction-B + 3× Inception-C) + classificador | Etapa 1: igual ao A. Etapa 2: **também** `features[11:18]` (7× Inception-B) |
| Learning rate | **1e-4** para tudo | Classificador **1e-4**, blocos descongelados **1e-5** |
| Etapas | 1 (até 20 épocas) | 2: até 5 épocas só com o topo, depois o restante do orçamento com o meio da rede também |
| Hipótese | Adaptar só as features de alto nível já basta e é barato | Adaptar também as features intermediárias (formas e partes) ajuda a separar animais parecidos (caranguejo × lagosta × camarão, foca × lontra), e a lr menor + descongelamento gradual protegem o pré-treino |
| Riscos | lr 1e-4 nos blocos pode "apagar" parte do pré-treino | Muito mais parâmetros: mais risco de overfitting, mais tempo e memória por época |

A comparação **base → A** mede o ganho de adaptar o topo da rede; **A → B** mede o ganho de ir mais fundo com cuidado. B muda duas coisas ao mesmo tempo (profundidade e learning rate), então a conclusão vale para a **estratégia como um todo**, não para cada fator isolado.

Parâmetros treináveis em cada condição:

In [ ]:
def blocos_treinaveis(model, condicao, etapa=2):
    freeze_base(model)
    if condicao in ("A", "B"):
        unfreeze(model.features[TOPO])
    if condicao == "B" and etapa == 2:
        unfreeze(model.features[MEIO])
    return model

total = count_params(base_model)[0]
plano = []
for nome, cond, etapa in [("Base congelada", "base", 0), ("A", "A", 1), ("B — etapa 1", "B", 1), ("B — etapa 2", "B", 2)]:
    blocos_treinaveis(base_model, cond, etapa)
    t = count_params(base_model)[1]
    plano.append({"condição": nome, "parâmetros treináveis": t, "% do total": round(100 * t / total, 1)})
freeze_base(base_model)   # devolve a base ao estado congelado
pd.DataFrame(plano).set_index("condição")

In [ ]:
def avaliar_e_salvar(model, cond, nome, info, hparams, camadas):
    # Mede na validação, salva checkpoint, previsões e resumo no mesmo formato da parte 2.
    va_loss, y, pred, prob = run_epoch(model, val_loader, criterion)
    m = compute_metrics(y, pred)
    total, trainable = count_params(model)
    torch.save({
        "model_name": MODEL_NAME, "num_classes": NUM_CLASSES, "drop_rate": DROP_RATE,
        "classes": classes, "class_to_idx": meta["class_to_idx"], "data_config": DATA_CFG,
        "state_dict": model.state_dict(), "condition": cond, "train_info": info,
        "hparams": hparams, "trainable_layers": camadas,
    }, ARTIFACTS / f"modelo_{cond}.pt")
    vdf = split[split.split == "val"].reset_index(drop=True).copy()
    vdf["pred"] = [classes[i] for i in pred]
    vdf["score"] = prob.max(1)
    vdf["p_real"] = prob[np.arange(len(y)), y]
    vdf["acertou"] = vdf.pred == vdf.label
    for i, c in enumerate(classes):
        vdf[f"prob_{c}"] = prob[:, i]
    vdf.to_csv(ARTIFACTS / f"val_preds_{cond}.csv", index=False)
    resumo = {"condicao": nome, "camadas_treinadas": camadas, "params_total": total, "params_treinaveis": trainable,
              **info, **{f"val_{k}": v for k, v in m.items()}, "val_loss": va_loss}
    json.dump(resumo, open(ARTIFACTS / f"resumo_{cond}.json", "w"), indent=2, ensure_ascii=False, default=float)
    print(f"{nome} | validação:", {k: round(v, 4) for k, v in m.items()}, f"| salvo em modelo_{cond}.pt")
    return resumo

## 3. Estratégia A — topo descongelado, etapa única

Carrega uma cópia do modelo de referência, descongela Reduction-B + 3× Inception-C (mantendo o classificador treinável) e treina tudo com lr 1e-4.

In [ ]:
seed_everything()
model_A, _ = load_checkpoint(BASE_CKPT)
blocos_treinaveis(model_A, "A")
print("Parâmetros (total, treináveis):", count_params(model_A))

opt_A = torch.optim.AdamW([p for p in model_A.parameters() if p.requires_grad], lr=LR_A, weight_decay=WEIGHT_DECAY)
sch_A = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_A, mode="max", factor=0.5, patience=2)
hist_A, info_A = fit(model_A, train_loader, val_loader, opt_A, sch_A, label="A")
hist_A.to_csv(ARTIFACTS / "history_A.csv", index=False)

resumo_A = avaliar_e_salvar(
    model_A, "A", "Estratégia A", info_A, camadas="features[18:] + classificador",
    hparams={"lr": LR_A, "weight_decay": WEIGHT_DECAY, "batch_size": BATCH_SIZE, "max_epochs": MAX_EPOCHS,
             "patience": PATIENCE, "monitor": MONITOR, "optimizer": "AdamW",
             "scheduler": "ReduceLROnPlateau(0.5, 2)", "seed": SEED})
plot_history(hist_A, "Estratégia A — topo descongelado (lr 1e-4)", info_A["best_epoch"])

**Análise das curvas da estratégia A** *(escrever com os resultados reais)*: comparar o patamar do F1 de validação com o da base congelada; observar se a loss de treino passa a cair bem abaixo da de validação (sinal de overfitting que não existia com a base congelada) e em que época o early stopping guardou o modelo.

## 4. Estratégia B — descongelamento progressivo com learning rate menor nos blocos

- **Etapa 1** (até 5 épocas): classificador (lr 1e-4) + topo (lr 1e-5).
- **Etapa 2** (restante das 20 épocas): também os 7 blocos Inception-B (lr 1e-5), partindo da melhor época da etapa 1.

Ao final fica a melhor época **das duas etapas** (se a etapa 2 não superar a etapa 1, os pesos da etapa 1 são mantidos).

In [ ]:
def grupos_B(model):
    cls = list(model.get_classifier().parameters())
    ids_cls = {id(p) for p in cls}
    blocos = [p for p in model.parameters() if p.requires_grad and id(p) not in ids_cls]
    return [{"params": cls, "lr": LR_B_CLASSIFICADOR}, {"params": blocos, "lr": LR_B_BLOCOS}]

seed_everything()
model_B, _ = load_checkpoint(BASE_CKPT)

# Etapa 1: topo
blocos_treinaveis(model_B, "B", etapa=1)
print("Etapa 1 — parâmetros (total, treináveis):", count_params(model_B))
opt_B1 = torch.optim.AdamW(grupos_B(model_B), weight_decay=WEIGHT_DECAY)
sch_B1 = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_B1, mode="max", factor=0.5, patience=2)
hist_B1, info_B1 = fit(model_B, train_loader, val_loader, opt_B1, sch_B1, max_epochs=B_ETAPA1_EPOCAS, label="B etapa 1")
estado_B1 = {k: v.detach().cpu().clone() for k, v in model_B.state_dict().items()}

# Etapa 2: topo + Inception-B, com o restante do orçamento de épocas
blocos_treinaveis(model_B, "B", etapa=2)
print("Etapa 2 — parâmetros (total, treináveis):", count_params(model_B))
opt_B2 = torch.optim.AdamW(grupos_B(model_B), weight_decay=WEIGHT_DECAY)
sch_B2 = torch.optim.lr_scheduler.ReduceLROnPlateau(opt_B2, mode="max", factor=0.5, patience=2)
hist_B2, info_B2 = fit(model_B, train_loader, val_loader, opt_B2, sch_B2,
                       max_epochs=MAX_EPOCHS - info_B1["epochs_run"], label="B etapa 2")

if info_B2["best_" + MONITOR] <= info_B1["best_" + MONITOR]:
    print("A etapa 2 não superou a etapa 1: mantendo os pesos da etapa 1.")
    model_B.load_state_dict(estado_B1)
    best_B = info_B1["best_epoch"]
else:
    best_B = info_B1["epochs_run"] + info_B2["best_epoch"]
del estado_B1

hist_B = pd.concat([hist_B1.assign(etapa=1),
                    hist_B2.assign(etapa=2, epoch=hist_B2.epoch + info_B1["epochs_run"])], ignore_index=True)
hist_B.to_csv(ARTIFACTS / "history_B.csv", index=False)
info_B = {"best_epoch": best_B, "epochs_run": len(hist_B),
          "best_" + MONITOR: max(info_B1["best_" + MONITOR], info_B2["best_" + MONITOR]),
          "train_time_s": info_B1["train_time_s"] + info_B2["train_time_s"],
          "epochs_etapa1": info_B1["epochs_run"], "epochs_etapa2": info_B2["epochs_run"]}

resumo_B = avaliar_e_salvar(
    model_B, "B", "Estratégia B", info_B, camadas="etapa 1: features[18:]; etapa 2: features[11:] + classificador",
    hparams={"lr_classificador": LR_B_CLASSIFICADOR, "lr_blocos": LR_B_BLOCOS, "epocas_etapa1": B_ETAPA1_EPOCAS,
             "weight_decay": WEIGHT_DECAY, "batch_size": BATCH_SIZE, "max_epochs": MAX_EPOCHS,
             "patience": PATIENCE, "monitor": MONITOR, "optimizer": "AdamW",
             "scheduler": "ReduceLROnPlateau(0.5, 2)", "seed": SEED})
plot_history(hist_B, "Estratégia B — progressiva (classificador 1e-4, blocos 1e-5)", best_B,
             marcos=[(info_B1["epochs_run"] + 0.5, "início da etapa 2")])

**Análise das curvas da estratégia B** *(escrever com os resultados reais)*: o salto (ou não) do F1 de validação ao entrar na etapa 2; se a distância treino × validação aumenta com mais blocos treináveis; quanto mais lenta ficou cada época.

## 5. Comparação das três condições (validação)

Todas as decisões desta seção usam **só a validação**. O teste continua intocado.

In [ ]:
resumo_base = json.load(open(ARTIFACTS / "resumo_base_congelada.json", encoding="utf-8"))
resumo_base["camadas_treinadas"] = "classificador"
hist_base = pd.read_csv(ARTIFACTS / "history_base_congelada.csv")

CONDS = {"base": ("Base congelada", resumo_base, hist_base),
         "A": ("Estratégia A", resumo_A, hist_A),
         "B": ("Estratégia B", resumo_B, hist_B)}

comp = pd.DataFrame([{
    "condição": nome,
    "camadas treinadas": r["camadas_treinadas"],
    "parâmetros treináveis": int(r["params_treinaveis"]),
    "épocas executadas": int(r["epochs_run"]),
    "melhor época": int(r["best_epoch"]),
    "tempo de treino (min)": r["train_time_s"] / 60,
    "F1-macro": r["val_f1_macro"],
    "acurácia balanceada": r["val_bal_acc"],
    "acurácia": r["val_acc"],
    "loss": r["val_loss"],
} for nome, r, _ in CONDS.values()]).set_index("condição")
comp.style.format({"parâmetros treináveis": "{:,}", "tempo de treino (min)": "{:.1f}", "F1-macro": "{:.4f}",
                   "acurácia balanceada": "{:.4f}", "acurácia": "{:.4f}", "loss": "{:.4f}"}) \
    .highlight_max(subset=["F1-macro", "acurácia balanceada", "acurácia"], color="#c6efce")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for (nome, _, h), cor in zip(CONDS.values(), ["#4C72B0", "#DD8452", "#55A868"]):
    axes[0].plot(h.epoch, h.val_loss, "o-", c=cor, label=f"{nome} — val")
    axes[0].plot(h.epoch, h.train_loss, "--", c=cor, alpha=.5, label=f"{nome} — treino")
    axes[1].plot(h.epoch, h.val_f1_macro, "o-", c=cor, label=f"{nome} — val")
    axes[1].plot(h.epoch, h.train_f1_macro, "--", c=cor, alpha=.5, label=f"{nome} — treino")
axes[0].set_title("Loss"); axes[1].set_title("F1-macro")
for ax in axes:
    ax.set_xlabel("época (cada condição a partir da sua 1ª época)"); ax.legend(fontsize=7)
fig.suptitle("Curvas das três condições (linha cheia = validação, tracejada = treino)")
plt.tight_layout(); plt.show()

> As curvas de A e B começam de onde a base congelada terminou (mesmos pesos iniciais), por isso já partem de um F1 alto. O eixo de épocas conta as épocas de **cada** condição.

In [ ]:
preds_val = {c: pd.read_csv(ARTIFACTS / f"val_preds_{c if c != 'base' else 'base_congelada'}.csv") for c in CONDS}
y_val = preds_val["base"].label.map(meta["class_to_idx"]).values
yhat_val = {c: d.pred.map(meta["class_to_idx"]).values for c, d in preds_val.items()}

fig, axes = plt.subplots(1, 3, figsize=(3 * (0.42 * NUM_CLASSES + 2.5), 0.42 * NUM_CLASSES + 2))
for ax, (c, (nome, _, _)) in zip(axes, CONDS.items()):
    cm = confusion_matrix(y_val, yhat_val[c], labels=range(NUM_CLASSES))
    cm_norm = cm / cm.sum(1, keepdims=True).clip(min=1)
    sns.heatmap(cm_norm, cmap="Blues", vmin=0, vmax=1, xticklabels=classes, yticklabels=classes,
                cbar=False, ax=ax, square=True)
    ax.set_title(f"{nome}\nF1-macro {f1_score(y_val, yhat_val[c], average='macro'):.4f}")
    ax.set_xlabel("prevista"); ax.set_ylabel("real")
    ax.tick_params(labelsize=7)
fig.suptitle("Matrizes de confusão na validação (cor = % da classe real)")
plt.tight_layout(); plt.show()

In [ ]:
# F1 por classe nas três condições: onde o fine-tuning ajudou (ou atrapalhou)
f1_cls = pd.DataFrame({nome: f1_score(y_val, yhat_val[c], labels=range(NUM_CLASSES), average=None, zero_division=0)
                       for c, (nome, _, _) in CONDS.items()}, index=classes)
f1_cls["Δ A − base"] = f1_cls["Estratégia A"] - f1_cls["Base congelada"]
f1_cls["Δ B − base"] = f1_cls["Estratégia B"] - f1_cls["Base congelada"]
f1_cls = f1_cls.sort_values("Base congelada")

ax = f1_cls[["Base congelada", "Estratégia A", "Estratégia B"]].plot.barh(figsize=(9, 0.35 * NUM_CLASSES + 1.5), width=.8)
ax.set_xlabel("F1 na validação"); ax.set_title("F1 por classe (ordenado pela base congelada)")
ax.set_xlim(max(0, f1_cls.iloc[:, :3].values.min() - 0.1), 1)
plt.tight_layout(); plt.show()
f1_cls.round(3)

In [ ]:
# Acompanhamento específico da classe Crabs (o "foco em caranguejos" da planilha)
if "Crabs" in classes:
    k = meta["class_to_idx"]["Crabs"]
    linhas = []
    for c, (nome, _, _) in CONDS.items():
        rep = classification_report(y_val, yhat_val[c], labels=range(NUM_CLASSES), target_names=classes,
                                    output_dict=True, zero_division=0)["Crabs"]
        erros = pd.Series(yhat_val[c][(y_val == k) & (yhat_val[c] != k)]).map(dict(enumerate(classes)))
        linhas.append({"condição": nome, "precisão": rep["precision"], "recall": rep["recall"], "F1": rep["f1-score"],
                       "Crabs previstos como": ", ".join(f"{a} ({n})" for a, n in erros.value_counts().head(3).items()) or "—"})
    display(pd.DataFrame(linhas).set_index("condição").round(3))

In [ ]:
# Modelo recomendado: maior F1-macro na VALIDAÇÃO (critério definido antes de ver o teste).
# Em caso de empate técnico (< 0,002), fica o mais simples (menos parâmetros treináveis).
ordem = comp.sort_values(["F1-macro", "parâmetros treináveis"], ascending=[False, True])
melhor = ordem.index[0]
for nome in ordem.index[1:]:
    if (ordem.loc[melhor, "F1-macro"] - ordem.loc[nome, "F1-macro"] < 0.002
            and ordem.loc[nome, "parâmetros treináveis"] < ordem.loc[melhor, "parâmetros treináveis"]):
        melhor = nome
COND_REC = {nome: c for c, (nome, _, _) in CONDS.items()}[melhor]
FT_REF = "A" if resumo_A["val_f1_macro"] >= resumo_B["val_f1_macro"] else "B"   # melhor fine-tuning, para a seção 6
print(f"Modelo recomendado pela validação: {melhor} (F1-macro {comp.loc[melhor, 'F1-macro']:.4f})")
print(f"Melhor fine-tuning na validação: {CONDS[FT_REF][0]}")

**Discussão** *(escrever com os resultados reais)*: o fine-tuning compensou em relação à base congelada? A diferença entre A e B justifica o custo extra de B (parâmetros, tempo por época)? Quais classes mais ganharam e quais pioraram? A classe Crabs melhorou em recall e deixou de ser confundida com Lobster/Shrimp?

## 6. Análise visual nas mesmas imagens

Comparamos, imagem por imagem da validação, a base congelada com o melhor fine-tuning:

- **Erros corrigidos:** a base errava e o fine-tuning acerta;
- **Erros persistentes:** os dois erram;
- **Erros novos:** a base acertava e o fine-tuning passou a errar.

Os títulos mostram a previsão dos **três** modelos (B = base, A, B̲ = estratégia B) com o score.

In [ ]:
vis = preds_val["base"][["path", "label"]].copy()
for c in CONDS:
    vis[f"pred_{c}"] = preds_val[c].pred.values
    vis[f"score_{c}"] = preds_val[c].score.values
    vis[f"ok_{c}"] = preds_val[c].acertou.values

tabela = []
for c in ["A", "B"]:
    tabela.append({"comparação": f"base → {CONDS[c][0]}",
                   "erros corrigidos": int((~vis.ok_base & vis[f"ok_{c}"]).sum()),
                   "erros persistentes": int((~vis.ok_base & ~vis[f"ok_{c}"]).sum()),
                   "erros novos": int((vis.ok_base & ~vis[f"ok_{c}"]).sum())})
tabela.append({"comparação": "erram nos três", "erros persistentes": int((~vis.ok_base & ~vis.ok_A & ~vis.ok_B).sum())})
display(pd.DataFrame(tabela).set_index("comparação").astype("Int64").astype(str).replace("<NA>", ""))

def titulos3(d):
    return [f"real: {r.label}\nbase: {r.pred_base} ({r.score_base:.2f})\nA: {r.pred_A} ({r.score_A:.2f})\n"
            f"B: {r.pred_B} ({r.score_B:.2f})" for r in d.itertuples()]

ok_ft = vis[f"ok_{FT_REF}"]
grupos = {
    "corrigidos": vis[~vis.ok_base & ok_ft].sort_values("score_base", ascending=False),
    "persistentes": vis[~vis.ok_base & ~ok_ft].sort_values(f"score_{FT_REF}", ascending=False),
    "novos": vis[vis.ok_base & ~ok_ft].sort_values(f"score_{FT_REF}", ascending=False),
}
for nome_g, d in grupos.items():
    if len(d):
        show_grid(d.path.head(6).tolist(), titulos3(d.head(6)),
                  suptitle=f"Erros {nome_g} ({CONDS[FT_REF][0]} vs base) — {len(d)} no total")
    else:
        print(f"Nenhum erro {nome_g}.")

**Leitura** *(completar)*: os erros corrigidos são de animais parecidos que agora são separados? Os persistentes têm algo em comum (vários animais na foto, animal pequeno, desenho, rótulo duvidoso)? Os erros novos são poucos e de baixa confiança, ou o fine-tuning "esqueceu" algo que a base sabia?

## 7. Grad-CAM: onde cada modelo está olhando

**Grad-CAM** (Selvaraju et al., *Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization*, ICCV 2017) pondera os mapas de ativação de uma camada convolucional pelo gradiente da classe de interesse e mostra as regiões da imagem que mais pesaram na decisão.

- **Camada:** `model.features[-1]`, o último bloco Inception-C (mapa 8×8×1536, a última camada espacial antes do pooling global). Por ser 8×8, o mapa é grosseiro: indica a **região**, não o contorno exato.
- **Classe explicada:** a classe **prevista** por cada modelo (o que fez aquele modelo decidir o que decidiu).
- **Imagens:** as mesmas nos três modelos — erros corrigidos, persistentes, novos e um acerto seguro.

Pergunta principal: o modelo olha para o **animal** ou para o **fundo** (água, areia, aquário)?

In [ ]:
def grad_cam(model, x, target=None):
    # x: tensor (1, 3, H, W) já normalizado. Retorna (mapa HxW em [0, 1], classe, score).
    model.eval()
    store = {}
    def hook(_, __, out):
        store["act"] = out
        out.register_hook(lambda g: store.__setitem__("grad", g))
    h = model.features[-1].register_forward_hook(hook)
    try:
        with torch.enable_grad():
            x = x.to(DEVICE).clone().requires_grad_(True)   # garante gradiente mesmo com a base congelada
            logits = model(x)
            prob = logits.softmax(1)[0].detach()
            target = int(prob.argmax()) if target is None else target
            model.zero_grad(set_to_none=True)
            logits[0, target].backward()
    finally:
        h.remove()
    w = store["grad"].mean(dim=(2, 3), keepdim=True)              # importância de cada canal
    cam = F.relu((w * store["act"]).sum(1, keepdim=True))           # (1, 1, 8, 8)
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    model.zero_grad(set_to_none=True)
    return cam.detach().cpu().numpy(), target, float(prob[target])

modelos = {"base": base_model, "A": model_A, "B": model_B}

escolha = pd.concat([
    grupos["corrigidos"].head(2).assign(grupo="corrigido"),
    grupos["persistentes"].head(2).assign(grupo="persistente"),
    grupos["novos"].head(1).assign(grupo="novo"),
    vis[vis.ok_base & vis.ok_A & vis.ok_B].sort_values("score_base", ascending=False).head(1).assign(grupo="acerto"),
])

fig, axes = plt.subplots(len(escolha), 4, figsize=(13, 3.4 * len(escolha)), squeeze=False)
for row, r in zip(axes, escolha.itertuples()):
    x = eval_tf(load_rgb(DATA_DIR / r.path))
    img = denorm(x).numpy()
    row[0].imshow(img); row[0].set_title(f"[{r.grupo}] real: {r.label}", fontsize=9)
    for ax, (c, m) in zip(row[1:], modelos.items()):
        cam, k, s = grad_cam(m, x[None])
        ax.imshow(img); ax.imshow(cam, cmap="jet", alpha=0.45)
        cor = "green" if classes[k] == r.label else "crimson"
        ax.set_title(f"{CONDS[c][0]}\nprev: {classes[k]} ({s:.2f})", fontsize=9, color=cor)
    for ax in row:
        ax.axis("off")
fig.suptitle("Grad-CAM (features[-1], classe prevista) — verde = acerto, vermelho = erro", y=1.0)
plt.tight_layout(); plt.show()

**Leitura do Grad-CAM** *(completar)*: o fine-tuning deslocou a atenção do fundo para o animal nos erros corrigidos? Nos persistentes, o mapa está sobre o animal (confusão "honesta" entre classes parecidas) ou sobre o fundo/outro objeto (atalho)? Nos acertos seguros os três modelos concordam na região?

## 8. Avaliação final no conjunto de teste (etapa 7)

O modelo recomendado já foi escolhido na seção 5, **pela validação**. Só agora o teste é usado, uma única vez, nos três modelos. Ele serve para estimar o desempenho em imagens novas, **não** para trocar a escolha.

In [ ]:
print("Modelo recomendado (escolhido antes do teste):", CONDS[COND_REC][0])

res_test, pred_test = [], {}
for c, m in modelos.items():
    te_loss, y_test, p, prob = run_epoch(m, test_loader, criterion)
    pred_test[c] = (p, prob)
    mt = compute_metrics(y_test, p)
    res_test.append({"condição": CONDS[c][0], "F1-macro teste": mt["f1_macro"], "acurácia balanceada teste": mt["bal_acc"],
                     "acurácia teste": mt["acc"], "loss teste": te_loss, "F1-macro validação": CONDS[c][1]["val_f1_macro"]})
res_test = pd.DataFrame(res_test).set_index("condição")
res_test["Δ F1 (teste − val)"] = res_test["F1-macro teste"] - res_test["F1-macro validação"]
res_test.to_csv(ARTIFACTS / "resultados_teste.csv")
res_test.round(4)

In [ ]:
p_rec = pred_test[COND_REC][0]
rep = pd.DataFrame(classification_report(y_test, p_rec, labels=range(NUM_CLASSES), target_names=classes,
                                         output_dict=True, zero_division=0)).T
display(rep.iloc[:NUM_CLASSES].sort_values("f1-score").round(3))

cm = confusion_matrix(y_test, p_rec, labels=range(NUM_CLASSES))
cm_norm = cm / cm.sum(1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(0.55 * NUM_CLASSES + 3, 0.5 * NUM_CLASSES + 2))
sns.heatmap(cm_norm, annot=cm, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes,
            cbar_kws={"label": "proporção da classe real"}, ax=ax, annot_kws={"size": 7})
ax.set_xlabel("classe prevista"); ax.set_ylabel("classe real")
ax.set_title(f"Matriz de confusão — TESTE, {CONDS[COND_REC][0]}")
plt.tight_layout(); plt.show()

conf = pd.DataFrame([(classes[i], classes[j], cm[i, j]) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES)
                     if i != j and cm[i, j] > 0], columns=["real", "prevista", "n"])
print("Principais confusões no teste:")
conf.sort_values("n", ascending=False).head(8)

**Conclusão** *(completar)*: o desempenho no teste confirma o da validação (Δ pequeno = a escolha não foi sorte da divisão)? A ordem das três condições se manteve? Limitações: holdout único (sem validação cruzada), duplicatas quase idênticas não detectadas pelo MD5, possíveis rótulos errados no dataset.

## 9. Modelo recomendado: salvar, carregar sem retreinar e prever uma imagem da internet

O checkpoint do modelo recomendado é copiado para `modelo_recomendado.pt`, carregado do zero (sem nenhum treino) e usado numa foto que **não faz parte do dataset**.

Preencha a URL de uma imagem de um animal de uma das classes, a fonte (site/autor/licença) e a classe esperada. Uma boa opção é o Wikimedia Commons (copiar o link do arquivo original, `upload.wikimedia.org/...`). Se a URL não funcionar, envie a imagem para o Colab e use `IMG_PATH`.

In [ ]:
rec_path = ARTIFACTS / "modelo_recomendado.pt"
origem = BASE_CKPT if COND_REC == "base" else ARTIFACTS / f"modelo_{COND_REC}.pt"
shutil.copyfile(origem, rec_path)
print(f"Copiado {origem.name} -> {rec_path.name} ({rec_path.stat().st_size / 1e6:.0f} MB)")

# Carrega do zero, sem retreinar, e confere que reproduz as previsões de teste
modelo_final, ck_final = load_checkpoint(rec_path)
_, _, p_check, _ = run_epoch(modelo_final, test_loader, criterion)
assert (p_check == p_rec).all(), "O modelo carregado não reproduz as previsões!"
print("Modelo carregado OK | condição:", ck_final["condition"], "| melhor época:", ck_final["train_info"]["best_epoch"])

In [ ]:
IMG_URL = ""            # ex.: link direto (upload.wikimedia.org/...) de uma foto de caranguejo, tartaruga etc.
IMG_PATH = ""           # alternativa: caminho de um arquivo enviado para o Colab
IMG_FONTE = ""          # ex.: "Wikimedia Commons, autor X, licença CC BY-SA 4.0, <link da página>"
CLASSE_ESPERADA = ""    # uma das classes abaixo

print("Classes do modelo:", ", ".join(ck_final["classes"]))

def carregar_imagem_externa():
    if IMG_PATH:
        return load_rgb(IMG_PATH)
    req = urllib.request.Request(IMG_URL, headers={"User-Agent": "Mozilla/5.0 (notebook academico)"})
    with urllib.request.urlopen(req, timeout=30) as resp:
        return load_rgb(io.BytesIO(resp.read()))

@torch.no_grad()
def prever(model, img, k=5):
    prob = model.eval()(eval_tf(img)[None].to(DEVICE)).float().softmax(1)[0].cpu()
    top = prob.topk(min(k, len(prob)))
    return [(ck_final["classes"][i], float(s)) for s, i in zip(top.values, top.indices)]

if not (IMG_URL or IMG_PATH):
    print("\n>> Preencha IMG_URL (ou IMG_PATH), IMG_FONTE e CLASSE_ESPERADA e rode esta célula de novo.")
else:
    assert CLASSE_ESPERADA in ck_final["classes"], "CLASSE_ESPERADA precisa ser uma das classes do modelo"
    img = carregar_imagem_externa()
    top5 = prever(modelo_final, img)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), gridspec_kw={"width_ratios": [1, 1, 1.2]})
    axes[0].imshow(img); axes[0].set_title(f"original ({img.width}×{img.height})"); axes[0].axis("off")
    x = eval_tf(img)
    cam, k, s = grad_cam(modelo_final, x[None])
    axes[1].imshow(denorm(x)); axes[1].imshow(cam, cmap="jet", alpha=0.45); axes[1].axis("off")
    axes[1].set_title(f"Grad-CAM — prev: {classes[k]} ({s:.2f})",
                      color="green" if classes[k] == CLASSE_ESPERADA else "crimson")
    nomes, scores = zip(*top5[::-1])
    axes[2].barh(nomes, scores, color=["#55A868" if n == CLASSE_ESPERADA else "#4C72B0" for n in nomes])
    axes[2].set_xlim(0, 1); axes[2].set_title("Top-5 (probabilidade)")
    fig.suptitle(f"Esperado: {CLASSE_ESPERADA} | Previsto: {top5[0][0]} (score {top5[0][1]:.3f})")
    plt.tight_layout(); plt.show()
    print("Fonte:", IMG_FONTE or IMG_URL or IMG_PATH)
    print("Classe esperada:", CLASSE_ESPERADA, "| previsão:", top5[0][0], f"| score {top5[0][1]:.3f}")
    print("Top-5:", ", ".join(f"{n} {s:.3f}" for n, s in top5))

---
**Arquivos gerados em `artifacts/`:** `modelo_A.pt`, `modelo_B.pt`, `modelo_recomendado.pt` (fora do git, ~165 MB cada — subir no Drive da equipe), `history_A.csv`, `history_B.csv`, `val_preds_A.csv`, `val_preds_B.csv`, `resumo_A.json`, `resumo_B.json`, `resultados_teste.csv`.